# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the [FAIR²](https://sen.science/doi/10.71728/senscience.vq4a-28xa/) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a [Croissant schema URL](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json).

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

> **Note:** According to the Croissant schema, each table or main data source is represented as a *record set*, which exposes *fields* (columns) uniquely by their `@id`.

In [ ]:
# Display all available record set @ids and their fields
print("Available Record Sets:")
record_sets = []
for rs in dataset.metadata.record_sets:
    print(f"- RecordSet @id: {rs.id}, name: {rs.name}")
    record_sets.append(rs.id)
    print("  Fields:")
    for f in rs.fields:
        print(f"    - Field @id: {f.id}, name: {f.name}, dataType: {f.data_type}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Below we choose the primary record set by its `@id` and extract the data using the `mlcroissant.Dataset.records` method.

Reference all entities by their `@id` field for consistency.

In [ ]:
# We'll use the first detected record set for demonstration (update as needed)
dataframes = {}

for record_set_id in record_sets:
    print(f"Loading records for RecordSet @id: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Columns: {df.columns.tolist()}")
    print(df.head(2))
    print("")
# For the remainder, use the first record set id
main_record_set_id = record_sets[0]
main_df = dataframes[main_record_set_id]

## 4. Exploratory Data Analysis (EDA)
Let's apply simple statistics and filtering on selected numeric fields from the main record set.

We'll select one numeric field (by `@id`) to:
- Filter values
- Normalize the field
- Group by a categorical field

In [ ]:
# List all available field @ids again for reference
print("All columns in the main record set:")
print(main_df.columns.tolist())
print("")
# Update these IDs after inspecting the columns above if different
# (Example IDs, replace with actual field @ids for analysis)

# Find the first numeric column and a group column
numeric_field = None
group_field = None
for rs in dataset.metadata.record_sets:
    if rs.id == main_record_set_id:
        for f in rs.fields:
            if f.data_type in ('Float', 'Integer', 'Number') and f.id in main_df.columns:
                numeric_field = f.id
                break
        for f in rs.fields:
            if f.data_type == 'Text' and f.id in main_df.columns:
                group_field = f.id
                break
        break

print(f"Using numeric field (@id): {numeric_field}")
print(f"Using group field (@id): {group_field}")

if numeric_field is not None and numeric_field in main_df.columns:
    # Drop missing values for numeric analysis
    main_df_num = main_df.dropna(subset=[numeric_field])

    # Try to convert to numeric type if not already
    main_df_num[numeric_field] = pd.to_numeric(main_df_num[numeric_field], errors='coerce')

    threshold = main_df_num[numeric_field].mean()
    filtered_df = main_df_num[main_df_num[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
    print(filtered_df.head())

    # Normalization
    filtered_df[f"{numeric_field}_normalized"] = (
        (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    )
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    if group_field is not None and group_field in main_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Grouped data by {group_field}, showing mean of {numeric_field}:")
        print(grouped_df.head())
else:
    print("No suitable numeric field was found with data.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset, such as the distribution of the main numeric field or grouping.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field is not None and numeric_field in main_df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(main_df_num[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    if group_field is not None and group_field in main_df_num.columns:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=main_df_num[group_field], y=main_df_num[numeric_field])
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to:
- Load a Croissant dataset using the `mlcroissant` library
- Inspect the dataset schema (record sets and fields by `@id`)
- Extract and analyze records with pandas
- Perform basic numeric filtering, normalization, and grouping
- Visualize record set field distributions

Continue by adjusting field and grouping selections to your analysis needs.